# Exercício 11 — Regressão com scikit-learn

Este exercício repete o pipeline da aula na **sua coleta de rede social** (a mesma que você usou nas Aulas 5 a 7). Você vai treinar um modelo que estima a taxa de engajamento de um post a partir de características que existem *antes* da publicação, e comparar esse modelo com um chute burro.

Não é para inventar coleta nova. Copie a sua `exportacao.csv` das aulas anteriores para `dados/exportacao.csv` dentro desta pasta.

Antes de tudo, copie a pasta `exercicios/` para dentro da sua pasta de entregas (`extracao-dados-trabalhos-seunome`, com o SEU nome), numa pasta `11-regressao-scikit-learn` dentro de `projetos/`.

## Preparação do ambiente

Este notebook mora numa pasta própria (dentro da sua pasta de entregas). Crie o ambiente e instale as dependências dentro dela.

No Windows (Prompt de Comando ou Terminal integrado do VS Code):

```cmd
uv venv .venv
uv pip install -r requirements.txt
```

No Mac (Terminal), os mesmos comandos. Se o `uv` não funcionar, use `pip install -r requirements.txt` com o ambiente já ativado.

## Parte 0 — Dados

**Fonte dos dados:** copie para `dados/exportacao.csv` a mesma coleta que você usou nas Aulas 5 a 7.

**Pergunta que este modelo tenta responder (preencha):**

> Consigo estimar a taxa de engajamento de um post da minha coleta a partir de características de uso de emojis, uso de hashtags e dia da semana em que o vídeo foi publicado?

## Parte 1 — Carregar e construir o alvo

Esse trecho já vem pronto (é o mesmo da aula). Ajuste os nomes de coluna se a sua plataforma usar nomes diferentes de `likes`, `comments`, `shares`, `plays`.

In [29]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("/Users/jmvianna/Documents/extracao-dados-joao/projetos/11-regressao-scikit-learn/dados/exportacao.csv", sep=";")
df = df.drop_duplicates()
df = df[df["plays"] > 0].copy()

df["taxa_engajamento"] = (
    df["likes"] + df["comments"] + df["shares"]
) / df["plays"]

y = df["taxa_engajamento"]
print(f"Posts na base: {len(df)}")
y.describe()

Posts na base: 1044


count    1044.000000
mean        0.061674
std         0.049655
min         0.000319
25%         0.030516
50%         0.050106
75%         0.077576
max         0.406375
Name: taxa_engajamento, dtype: float64

## Parte 2 — Construir as suas features

**Regra de ouro: nada de vazamento.** `likes`, `comments`, `shares` e `plays` NÃO entram no `X` — eles são o alvo.

Monte um `X` com **pelo menos três features derivadas por você**. Pode reaproveitar as da aula (nº de hashtags, tamanho da legenda, nº de emojis, hora, dia da semana, seguidores do autor) e/ou inventar outras que façam sentido para a sua coleta (tem menção `@`? tem link? tem ponto de interrogação? é fim de semana?).

In [30]:
import re

X = pd.DataFrame(index=df.index)

# complete aqui: crie pelo menos TRÊS colunas de feature em X.
# exemplos da aula (use, troque ou acrescente):
# X["seguidores_autor"] = df["author_followers"]
# X["tam_legenda"] = df["body"].fillna("").str.len()
# X["n_hashtags"] = df["hashtags"].fillna("").apply(lambda s: 0 if s == "" else len(s.split(",")))
# momento = pd.to_datetime(df["timestamp"])
# X["hora"] = momento.dt.hour
padrao_emoji = re.compile("[\U0001F000-\U0001FAFF\u2600-\u27BF]")  # faixa Unicode de emojis
X["n_emojis"] = df["body"].fillna("").apply(lambda s: len(padrao_emoji.findall(s)))
X["n_hashtags"] = df["hashtags"].fillna("").apply(
    lambda s: 0 if s == "" else len(s.split(","))
)

momento = pd.to_datetime(df["timestamp"])  # converte o texto de data para datetime
X["dia_semana"] = momento.dt.dayofweek  # 0 = segunda ... 6 = domingo


print("Features criadas:", list(X.columns))
X.head()

Features criadas: ['n_emojis', 'n_hashtags', 'dia_semana']


,n_emojis,n_hashtags,dia_semana
0,0,2,1
1,0,5,6
2,1,5,0
3,0,5,5
4,2,4,3


**Confira antes de seguir:** todas as colunas de `X` são numéricas? Nenhuma é `likes`/`comments`/`shares`/`plays`? Nenhuma tem valor ausente? (`X.isna().sum()` e `X.dtypes`)

In [31]:
print(X.dtypes)
print()
print("valores ausentes por coluna:")
print(X.isna().sum())

n_emojis      int64
n_hashtags    int64
dia_semana    int32
dtype: object

valores ausentes por coluna:
n_emojis      0
n_hashtags    0
dia_semana    0
dtype: int64


## Parte 3 — Treino, teste e modelo bobo

In [32]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.25, random_state=42
)

# modelo bobo: prever sempre a média do treino
previsao_boba = np.full(len(y_teste), y_treino.mean())
mae_bobo = mean_absolute_error(y_teste, previsao_boba)
r2_bobo = r2_score(y_teste, previsao_boba)

# regressão linear
modelo_linear = LinearRegression()
modelo_linear.fit(X_treino, y_treino)
previsoes = modelo_linear.predict(X_teste)
mae_linear = mean_absolute_error(y_teste, previsoes)
r2_linear = r2_score(y_teste, previsoes)

print(f"Modelo bobo:       MAE = {mae_bobo:.4f}   R2 = {r2_bobo:.3f}")
print(f"Regressão linear:  MAE = {mae_linear:.4f}   R2 = {r2_linear:.3f}")

Modelo bobo:       MAE = 0.0314   R2 = -0.002
Regressão linear:  MAE = 0.0319   R2 = -0.007


## Parte 4 — Uma árvore, para comparar

Complete a célula: treine uma `DecisionTreeRegressor` (use `max_depth=5, random_state=42`), preveja em `X_teste` e calcule MAE e R². Imprima os três modelos lado a lado (bobo, linear, árvore).

In [33]:
from sklearn.tree import DecisionTreeRegressor

# complete aqui: crie, treine (fit), preveja (predict) e meça (mae/r2) a árvore

modelo_arvore = DecisionTreeRegressor(max_depth=5, random_state=42)
modelo_arvore.fit(X_treino, y_treino)
previsoes_arvore = modelo_arvore.predict(X_teste)

mae_arvore = mean_absolute_error(y_teste, previsoes_arvore)
r2_arvore = r2_score(y_teste, previsoes_arvore)

print(f"Modelo bobo:        MAE = {mae_bobo:.4f}   R2 = {r2_bobo:.3f}")
print(f"Regressão linear:   MAE = {mae_linear:.4f}   R2 = {r2_linear:.3f}")
print(f"Árvore (prof. 5):   MAE = {mae_arvore:.4f}   R2 = {r2_arvore:.3f}")

Modelo bobo:        MAE = 0.0314   R2 = -0.002
Regressão linear:   MAE = 0.0319   R2 = -0.007
Árvore (prof. 5):   MAE = 0.0325   R2 = -0.048


In [34]:
from sklearn.tree import DecisionTreeRegressor

# complete aqui: crie, treine (fit), preveja (predict) e meça (mae/r2) a árvore

modelo_arvore = DecisionTreeRegressor(max_depth=4, random_state=42)
modelo_arvore.fit(X_treino, y_treino)
previsoes_arvore = modelo_arvore.predict(X_teste)

mae_arvore = mean_absolute_error(y_teste, previsoes_arvore)
r2_arvore = r2_score(y_teste, previsoes_arvore)

print(f"Modelo bobo:        MAE = {mae_bobo:.4f}   R2 = {r2_bobo:.3f}")
print(f"Regressão linear:   MAE = {mae_linear:.4f}   R2 = {r2_linear:.3f}")
print(f"Árvore (prof. 5):   MAE = {mae_arvore:.4f}   R2 = {r2_arvore:.3f}")

Modelo bobo:        MAE = 0.0314   R2 = -0.002
Regressão linear:   MAE = 0.0319   R2 = -0.007
Árvore (prof. 5):   MAE = 0.0322   R2 = -0.031


## Parte 5 — Ler os coeficientes

Complete: monte um DataFrame com `X.columns` e `modelo_linear.coef_`, ordenado pelo coeficiente. Olhe o sinal de cada um e escreva, no rascunho abaixo, **uma** leitura em uma frase (lembrando: associação nesta coleta, não causa).

In [35]:
# complete aqui: DataFrame com feature e coeficiente, ordenado
coeficientes = pd.DataFrame({
    "feature": X.columns,
    "coeficiente": modelo_linear.coef_,
}).sort_values("coeficiente")

coeficientes

,feature,coeficiente
1,n_hashtags,-0.000878
0,n_emojis,-0.000304
2,dia_semana,0.000446


In [36]:
from sklearn.preprocessing import StandardScaler

# padroniza as features (média 0, desvio padrão 1)
scaler = StandardScaler()
X_treino_padronizado = scaler.fit_transform(X_treino)
X_teste_padronizado = scaler.transform(X_teste)

# treina a regressão linear com as features padronizadas
modelo_linear_padronizado = LinearRegression()
modelo_linear_padronizado.fit(X_treino_padronizado, y_treino)

# monta a tabela de coeficientes, agora comparáveis
coeficientes_padronizados = pd.DataFrame({
    "feature": X.columns,
    "coeficiente_padronizado": modelo_linear_padronizado.coef_,
}).sort_values("coeficiente_padronizado", ascending=False)

coeficientes_padronizados

,feature,coeficiente_padronizado
2,dia_semana,0.000848
0,n_emojis,-0.000849
1,n_hashtags,-0.003935


Padronizei as variáveis porque "n_emojis" e "n_hashtags" estão em escalas diferentes. Depois da padronização, cada coeficiente passa a significar "quanto o engajamento muda para um desvio padrão de variação daquela variável", permitindo comparar de fato qual delas pesa mais na previsão.

**Rascunho da leitura de um coeficiente (vai para o README):**

> Mudei a base de dados, refazer a análise!Nesta coleta, quanto mais emojis inseridos a taxa de engajamento tende a ser maior ("n_emojis" maior coeficiente entre os analisados). Nesse sentido, a cada 1 emoji adicional a taxa de engajemnto prevista aumenta 0,007849 (cerca de 0,78 ponto percentual). Enquanto para a variável das hashtags ("n_hashtags") a mesma interpretação pode ser feita, a cada hashtag adicional a taxa de engajamento aumenta 0.001659 (0,17 ponto percentual). Entretanto, é válido esclarecer que esse comportamento é uma associação nos dados da amostra, não quer em todos os vídos, adicionar mais emojis ou hashtags de fato aumenta o engajamento.

## Parte 6 — README de reprodução

Crie o arquivo `README.md` dentro de `projetos/11-regressao-scikit-learn/` na sua pasta de entregas, respondendo:

**Fonte e período dos dados:**

> Dados de vídeos do Tiktok coletados através do Zeeschuimer

**Quantos posts entraram no modelo (depois de remover duplicata e `plays` = 0):**

> 640

**Quais features você usou, e por quê:**

> "n_hashtags" pois queria entender se o uso incremental de hashtags alterava a taxa de engajamento. "dia_semana" pra entender se postar em dias diferentes altera a taxa de engajamento (positivamente ou negativamente). E "n_emojis"para entender se o uso de emojis em legendas de vídeos nfluencia na taxa de engajamento nos materiais dessa amostra.

**Quais colunas você deixou de fora por vazamento, e por quê:**

> "likes", "comments", "shares", "plays" pois eles são exatamente as features que configuram a medição da taxa de engajamento.

**Resultado: MAE e R² do modelo bobo, da linear e da árvore. O seu melhor modelo bateu o bobo?**

> Escreva aqui.

**Uma leitura de coeficiente (associação, não causa):**

> Escreva aqui.

**Declaração de uso de IA:** ferramenta usada, em que trecho ou decisão desta entrega, e o que você conferiu ou alterou depois do resultado gerado (mesmo que a resposta seja "não usei IA nesta entrega", registre isso).

> Escreva aqui.

## Parte 7 — Conferência final

Antes de considerar a entrega concluída, confira:

- [ ] `dados/exportacao.csv` é a sua coleta das Aulas 5 a 7, e a pasta `dados/` está no `.gitignore`.
- [ ] O `X` tem pelo menos três features derivadas por você, todas numéricas, nenhuma delas `likes`/`comments`/`shares`/`plays`.
- [ ] A regressão linear e a árvore foram treinadas e comparadas com o modelo bobo.
- [ ] O README tem: fonte, nº de posts, lista de features, colunas descartadas por vazamento, os três resultados (MAE/R²) e uma leitura de coeficiente.
- [ ] Este notebook roda do início ao fim sem erro com Kernel → Restart e Run All (ou o equivalente no VS Code).
- [ ] O README registra o uso de IA (ou informa que não houve).
- [ ] Este notebook e o README estão copiados dentro de `projetos/11-regressao-scikit-learn/` na sua pasta de entregas.
- [ ] Você já fez `git add`, `git commit` e `git push` dessa entrega.